# Biblical Llama 3.1 Fine-Tuning with Unsloth (4-bit QLoRA v2)

**Base Model:** Llama 3.1 8B Instruct

**Dataset:** Augmentoolkit-generated Biblical persona (first-person apostolic responses)

**Key Difference from v1:** This version uses 4-bit QLoRA training and saves directly to 4-bit quantized format, avoiding post-training quantization issues.

**Chat Template:** `<|begin_of_text|><|start_header_id|>role<|end_header_id|>content<|eot_id|>`

## Step 1: Configuration

All paths and variables for easy configuration.

In [1]:
# ============================================================================
# CONFIGURATION - All variables for easy setup
# ============================================================================

# =========================== MODEL CONFIGURATION ===========================
# Base model to fine-tune (use Unsloth's pre-quantized 4-bit model for faster loading)
BASE_LLM = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

# Name for output folders and model identification
MODEL_NAME_BASE = "biblical_llama31_8b_instruct_unsloth_4bit"

# =========================== INPUT DATA ===========================
# Path to training data (Augmentoolkit output)
INPUT_DATA_PATH = "/home/spark/projects/augmentoolkit/outputs/biblical_persona_dataset"

# =========================== OUTPUT DIRECTORIES ===========================
# All outputs organized under ./output/{MODEL_NAME_BASE}/
OUTPUT_BASE_DIR = f"./output/{MODEL_NAME_BASE}"
OUTPUT_DIR_ADAPTERS = f"{OUTPUT_BASE_DIR}/train"       # LoRA weights during training
OUTPUT_DIR_MERGED = f"{OUTPUT_BASE_DIR}/model_4bit"    # 4-bit merged model
OUTPUT_DIR_GGUF = f"{OUTPUT_BASE_DIR}/gguf"            # GGUF for Ollama

# =========================== TRAINING HYPERPARAMETERS ===========================
MAX_SEQ_LENGTH = 2048    # Max tokens per example
BATCH_SIZE = 4           # Per-device batch size (can be higher with 4-bit!)
GRAD_ACCUM = 4           # Gradient accumulation
LEARNING_RATE = 2e-4     # Learning rate (can be slightly higher with QLoRA)
TARGET_EPOCHS = 1        # Number of training epochs

# =========================== LoRA CONFIGURATION ===========================
# LoRA (Low-Rank Adaptation) allows efficient fine-tuning by only training
# small adapter matrices instead of full model weights.
#
# For 4-bit QLoRA, we can use slightly higher ranks since memory is saved
#
# LORA_R (Rank): Controls adapter capacity
#   - Lower (4-8): Less aggressive, preserves base model behavior
#   - Higher (16-64): More capacity for new knowledge, risk of overfitting
#   - Recommendation: 16 for QLoRA (more room due to 4-bit base)
#
# LORA_ALPHA: Scaling factor for LoRA weights
#   - Typically set to 2x LORA_R (e.g., r=16 → alpha=32)
#   - Higher alpha = stronger influence of fine-tuning
#
# LORA_DROPOUT: Regularization during training
#   - 0: No dropout (faster training)
#   - 0.05-0.1: Mild regularization for larger datasets
#
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0

# =========================== LoRA TARGET MODULES ===========================
# Which layers to fine-tune. Llama architecture has these trainable layers:
#
# ATTENTION MODULES (Recommended for persona/style transfer)
# ┌──────────────┬─────────────────────────┬─────────────────────────────────────┐
# │ Module       │ Function                │ Training Impact                     │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ q_proj       │ Query projection        │ "What am I looking for?"            │
# │              │                         │ Core attention steering             │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ k_proj       │ Key projection          │ "What info do I have?"              │
# │              │                         │ Memory/context matching             │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ v_proj       │ Value projection        │ "What info to pass forward?"        │
# │              │                         │ Content selection                   │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ o_proj       │ Output projection       │ Combines attention heads            │
# │              │                         │ Post-attention mixing               │
# └──────────────┴─────────────────────────┴─────────────────────────────────────┘
#
# MLP/FFN MODULES (More aggressive - use for knowledge injection)
# ┌──────────────┬─────────────────────────┬─────────────────────────────────────┐
# │ Module       │ Function                │ Training Impact                     │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ gate_proj    │ Gate projection         │ Controls information flow           │
# │              │                         │ Heavy knowledge modification        │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ up_proj      │ Up projection           │ Expands to hidden dimension         │
# │              │                         │ Feature extraction                  │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ down_proj    │ Down projection         │ Compresses back to model dim        │
# │              │                         │ Output generation                   │
# └──────────────┴─────────────────────────┴─────────────────────────────────────┘
#
# PRESETS:
#   Conservative (style/persona):  ["q_proj", "v_proj"]
#   Balanced:                      ["q_proj", "k_proj", "v_proj", "o_proj"]
#   Aggressive (new knowledge):    ["q_proj", "k_proj", "v_proj", "o_proj", 
#                                   "gate_proj", "up_proj", "down_proj"]
#
LORA_TARGET_MODULES = ["q_proj", "v_proj"]  # Conservative: attention-only

# =========================== GGUF CONVERSION ===========================
# Path to llama.cpp installation (for GGUF conversion)
LLAMA_CPP_PATH = "/home/spark/resources/llama.cpp"

# Quantization type for GGUF export
# Options: None (fp32), "q4", "q8", "q4_k", "q5_k", "q6_k", "fp16"
# Recommendation: "q4_k" for 4-bit models (matches training precision)
QUANTIZATION_TYPE = "q4_k"

# =========================== VLLM DEPLOYMENT ===========================
# Centralized vLLM models directory (mapped to /models in container)
VLLM_MODELS_DIR = "/home/spark/projects/stoic/output/vllm"

# =========================== INFERENCE TEST ===========================
# Test prompt for inference validation
TEST_PROMPT = "I am struggling with forgiveness. What does Scripture teach about forgiving others?"

# ============================================================================
print("✓ Configuration loaded (4-bit QLoRA version)")
print(f"  Base model: {BASE_LLM}")
print(f"  Model name: {MODEL_NAME_BASE}")
print(f"  Input data: {INPUT_DATA_PATH}")
print(f"  Output base: {OUTPUT_BASE_DIR}")
print(f"  LoRA config: r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGET_MODULES}")
print(f"  Training precision: 4-bit QLoRA")

✓ Configuration loaded (4-bit QLoRA version)
  Base model: unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit
  Model name: biblical_llama31_8b_instruct_unsloth_4bit
  Input data: /home/spark/projects/augmentoolkit/outputs/biblical_persona_dataset
  Output base: ./output/biblical_llama31_8b_instruct_unsloth_4bit
  LoRA config: r=16, alpha=32, targets=['q_proj', 'v_proj']
  Training precision: 4-bit QLoRA


## 2. Environment Preparation

Install Unsloth and updated HuggingFace libraries.

In [2]:
# Install core packages from PyPI (much faster than git installs)
!pip install -q unsloth transformers trl peft accelerate datasets bitsandbytes

# Verify installations
import unsloth
import transformers
import trl
print(f"✓ Unsloth: {unsloth.__version__}")
print(f"✓ Transformers: {transformers.__version__}")
print(f"✓ TRL: {trl.__version__}")
print("Environment ready!")


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/spark/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/spark/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()


🦥 Unsloth Zoo will now patch everything to make training faster!
✓ Unsloth: 2026.1.4
✓ Transformers: 4.57.3
✓ TRL: 0.24.0
Environment ready!


## 3. Load Dataset & Format for Instruction Tuning

Load the Augmentoolkit-generated Biblical dataset (first-person apostolic responses from Paul, David, Peter, etc.).

In [3]:
from datasets import load_dataset, concatenate_datasets
import glob

# Load ALL subdirectories and ALL files
all_dirs = glob.glob(f"{INPUT_DATA_PATH}/*/")

print("📚 LOADING ALL AUGMENTOOLKIT DATA")
print(f"Found {len(all_dirs)} subdirectories")

datasets = []
for dir_path in sorted(all_dirs):
    jsonl_files = glob.glob(f"{dir_path}/*.jsonl")
    for file_path in jsonl_files:
        try:
            ds = load_dataset("json", data_files=file_path, split="train")
            datasets.append(ds)
            print(f"  Loaded {len(ds)} from {dir_path.split('/')[-2]}/{file_path.split('/')[-1]}")
        except Exception as e:
            print(f"  Skipped {file_path.split('/')[-1]}: {e}")

dataset = concatenate_datasets(datasets)

print(f"\n✓ Total dataset loaded: {len(dataset)} examples")
print(f"✓ Columns: {dataset.column_names}")
print(f"\n--- First Example (first 800 chars) ---")
import json
print(json.dumps(dataset[0], indent=2)[:800])

# Shuffle dataset for better training
dataset = dataset.shuffle(seed=42)
print(f"\n✓ Dataset shuffled and ready for training")

📚 LOADING ALL AUGMENTOOLKIT DATA
Found 50 subdirectories
  Loaded 123 from factual_sft_full_biblical_data_followup_0/simplified_data_rag.jsonl
  Loaded 123 from factual_sft_full_biblical_data_followup_0/plain_qa_list.jsonl
  Loaded 114 from factual_sft_full_biblical_data_followup_1/simplified_data_rag.jsonl
  Loaded 114 from factual_sft_full_biblical_data_followup_1/plain_qa_list.jsonl
  Loaded 121 from factual_sft_full_biblical_data_followup_2/simplified_data_rag.jsonl
  Loaded 121 from factual_sft_full_biblical_data_followup_2/plain_qa_list.jsonl
  Loaded 113 from factual_sft_full_biblical_data_followup_3/simplified_data_rag.jsonl
  Loaded 113 from factual_sft_full_biblical_data_followup_3/plain_qa_list.jsonl
  Loaded 114 from factual_sft_full_biblical_data_followup_4/simplified_data_rag.jsonl
  Loaded 114 from factual_sft_full_biblical_data_followup_4/plain_qa_list.jsonl
  Loaded 112 from factual_sft_full_biblical_data_followup_5/simplified_data_rag.jsonl
  Loaded 112 from factual_s

Generating train split: 0 examples [00:00, ? examples/s]

  Skipped Augmentoolkit-Augmentoolkit-Pippa-Thoughts_0.jsonl: An error occurred while generating the dataset



Generating train split: 0 examples [00:00, ? examples/s]


  Skipped Augmentoolkit-Augmentoolkit-Bluemoon-1mil-thoughts_0.jsonl: An error occurred while generating the dataset


...

  Skipped Augmentoolkit-Augmentoolkit-LMsys-800k-Thoughts_0.jsonl: An error occurred while generating the dataset


Generating train split: 0 examples [00:00, ? examples/s]


  Skipped Augmentoolkit-Openthoughts-100mil-DifferentFormat_0.jsonl: An error occurred while generating the dataset


Generating train split: 0 examples [00:00, ? examples/s]


  Skipped Augmentoolkit-Augmentoolkit-Capybara-2point5mil-Thoughts_0.jsonl: An error occurred while generating the dataset
  Loaded 2309 from inferred_facts_full_biblical_data/final_output.jsonl
  Loaded 26 from pretraining_run/text_chunks_full_biblical_data.jsonl
  Loaded 2199 from pretraining_run/representation_variation_full_biblical_data.jsonl
  Loaded 2309 from pretraining_run/inferred_facts_full_biblical_data.jsonl
  Loaded 134 from rag_data_full_biblical_data/axolotl_rag_conversations.jsonl
  Loaded 5932 from rag_source_data/rag_data_full_biblical_data.jsonl
  Loaded 5932 from rag_source_data/rag_data_combined.jsonl
  Loaded 2199 from representation_variation_full_biblical_data/final_output.jsonl
  Loaded 26 from sft_run/pretraining_subset_441912.jsonl
  Loaded 134 from sft_run/axolotl_rag_conversations_full_biblical_data.jsonl

✓ Total dataset loaded: 30274 examples
✓ Columns: ['conversations', 'text', 'metadata', 'segments', 'question', 'source_text', 'source_metadata', 'relat

## 4. Load Model & Tokenizer with Unsloth (4-bit)

Load Llama 3.1 8B Instruct model in **4-bit precision** for QLoRA training.

**Benefits of 4-bit training:**
- ~4x less VRAM usage
- Can use larger batch sizes
- Direct 4-bit output (no post-quantization needed)
- Faster training on consumer GPUs

In [4]:
from unsloth import FastLanguageModel
import torch

# Load model in 4-bit precision for QLoRA training
model, tokenizer = FastLanguageModel.from_pretrained(
    BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,           # Auto-detect (will use bfloat16 for compute)
    load_in_4bit=True,    # 4-bit quantization for QLoRA
    device_map="auto"     # Automatic device placement
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"✓ Model loaded: {BASE_LLM}")
print(f"✓ Precision: 4-bit (QLoRA)")
print(f"✓ Tokenizer configured")
print(f"✓ Max sequence length: {MAX_SEQ_LENGTH}")

==((====))==  Unsloth 2026.1.4: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 119.697 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
✓ Model loaded: unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit
✓ Precision: 4-bit (QLoRA)
✓ Tokenizer configured
✓ Max sequence length: 2048


In [5]:
# Format dataset for Llama 3.1 chat template
# Handle BOTH ShareGPT format (conversations) AND raw text format
def format_instruct(example):
    # If has conversations field AND it's not null, convert ShareGPT to chat template
    if example.get("conversations") is not None:
        messages = []
        for turn in example["conversations"]:
            # ShareGPT uses "from": "system"/"human"/"gpt"
            # Standard uses "role": "system"/"user"/"assistant"
            if turn["from"] == "system":
                messages.append({"role": "system", "content": turn["value"]})
            elif turn["from"] == "human":
                messages.append({"role": "user", "content": turn["value"]})
            elif turn["from"] == "gpt":
                messages.append({"role": "assistant", "content": turn["value"]})
        
        text = tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=False
        )
        return {"text": text}
    
    # Otherwise if it has a text field (raw text files), keep it as-is
    elif example.get("text") is not None and len(str(example["text"])) > 0:
        return {"text": str(example["text"])}
    
    # Skip malformed examples
    return {"text": ""}

# Format and keep only text column
dataset = dataset.map(format_instruct, remove_columns=dataset.column_names)

# Filter out empty examples
dataset = dataset.filter(lambda x: len(x["text"]) > 0)

print(f"\n--- Sample formatted text (first 500 chars) ---")
print(dataset[0]['text'][:500])
print(f"\n✓ Dataset formatted: {len(dataset)} examples")


--- Sample formatted text (first 500 chars) ---
[[[OVERALL_CONTEXT_IS -> Biblical Faith and Apostolic Teaching]]]
Specific source: habakkuk.txt
The ultimate consequence of Habakkuk ' s vision is that the Lord God is his strength, andhe will make HahaukKk 's feet likehinds ' fSst, WnaGlinN him to walk upon his high places. This is the culmination of a series of events that began with Habakkuk ' scryto God, feeling that God was not hearing him. However, God had already set in motion a plan to worka work in the days of Habakkuk ' s audience that

✓ Dataset formatted: 18142 examples


## 5. Add LoRA Adapters

Configure LoRA for efficient fine-tuning. See configuration section for module explanations.

In [6]:
from peft import LoraConfig

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH
)

print(f"✓ LoRA adapters added (r={LORA_R}, targets={LORA_TARGET_MODULES})")
print(f"✓ Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch Attention layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch O projection layer with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.1.4 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


✓ LoRA adapters added (r=16, targets=['q_proj', 'v_proj'])
✓ Trainable parameters: 6,815,744


## 6. Trainer Setup & Training

**PURE DOMAIN DATA with LIGHT TRAINING:**
- 100% authentic Biblical examples from epistles and psalms
- 1 epoch with low learning rate (2e-4) to gently teach persona
- This preserves base model capabilities while adding authentic voice

In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Calculate training steps
effective_batch_size = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = len(dataset) // effective_batch_size
max_steps = steps_per_epoch * TARGET_EPOCHS
warmup_steps = max(1, max_steps // 10)
save_steps = min(200, steps_per_epoch)  # Save every 200 steps or at epoch end

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=warmup_steps,
        max_steps=max_steps,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR_ADAPTERS,
        report_to="none",
        save_strategy="steps",
        save_steps=save_steps,
    )
)

print("✓ Trainer configured for 4-bit QLoRA training")
print(f"✓ Dataset size: {len(dataset)} conversations")
print(f"✓ Effective batch size: {effective_batch_size} (batch={BATCH_SIZE} × grad_accum={GRAD_ACCUM})")
print(f"✓ Steps per epoch: {steps_per_epoch}")
print(f"✓ Total epochs: {TARGET_EPOCHS}")
print(f"✓ Total steps: {max_steps}")
print(f"✓ Warmup steps: {warmup_steps}")
print(f"✓ Save every: {save_steps} steps (every epoch)")

✓ Trainer configured for 4-bit QLoRA training
✓ Dataset size: 18142 conversations
✓ Effective batch size: 16 (batch=4 × grad_accum=4)
✓ Steps per epoch: 1133
✓ Total epochs: 1
✓ Total steps: 1133
✓ Warmup steps: 113
✓ Save every: 200 steps (every epoch)


In [8]:
# Start training
trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 18,142 | Num Epochs = 1 | Total steps = 1,133
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 6,815,744 of 8,037,076,992 (0.08% trained)


Step,Training Loss
10,1.585500
20,1.617600
30,1.518100
40,1.380500
50,1.220900
60,1.226600
70,1.006800
80,1.033600
90,0.989400
100,1.041600


TrainOutput(global_step=1133, training_loss=0.8489166457150115, metrics={'train_runtime': 8507.1551, 'train_samples_per_second': 2.131, 'train_steps_per_second': 0.133, 'total_flos': 6.929075744804045e+17, 'train_loss': 0.8489166457150115, 'epoch': 0.9991181657848325})

## 7. Save LoRA Adapters

Save the trained LoRA adapters separately. These can be loaded on any Llama 3.1 8B model later.

**This is the primary output** - merging to full model is optional (Step 8).

In [9]:
from pathlib import Path

# Save LoRA adapters (primary output)
LORA_OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/lora_adapters"
Path(LORA_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"💾 Saving LoRA adapters to {LORA_OUTPUT_DIR}...")
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)

print(f"✅ LoRA adapters saved!")
print(f"   Path: {LORA_OUTPUT_DIR}")
print(f"   Can be loaded on any Llama 3.1 8B model with PEFT")

💾 Saving LoRA adapters to ./output/biblical_llama31_8b_instruct_unsloth_4bit/lora_adapters...
✅ LoRA adapters saved!
   Path: ./output/biblical_llama31_8b_instruct_unsloth_4bit/lora_adapters
   Can be loaded on any Llama 3.1 8B model with PEFT
